In [9]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=FutureWarning)

DATA_PATH = Path("..") / "DS0001" / "38304-0001-Data.tsv"
df = pd.read_csv(DATA_PATH, sep="\t")
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")

Loaded 1,495 rows, 234 columns


In [26]:
def zscore(s: pd.Series) -> pd.Series:
    """
    Standardize variables to have mean 0 and standard deviation 1
    """
    s = pd.to_numeric(s, errors="coerce")
    mu, sd = s.mean(), s.std(ddof=0)
    if sd == 0 or np.isnan(sd):
        return s * np.nan
    return (s - mu) / sd


d = df.copy()

# --- Core numeric / indicators ---
d["widowed"] = (pd.to_numeric(d["MARITAL"], errors="coerce") == 5).astype(float)
d["sbq_total"] = pd.to_numeric(d["SBQ_TOTAL"], errors="coerce")
d["stress_count"] = pd.to_numeric(d["STRESS_COUNT"], errors="coerce")
d["combat"] = pd.to_numeric(d["CES_TOTAL"], errors="coerce")
d["CES_TOTAL"] = d["combat"]

# Loneliness (three-item scale mean; higher = more lonely)
d["loneliness"] = pd.to_numeric(d["LON_MEAN"], errors="coerce")

# Social support (WHOQOL): higher = more satisfied / more support
support_cols = ["WHO20", "WHO22"]
d["social_support"] = d[support_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)

# Optimism (wellness): higher = more optimistic
optim_cols = ["WELLNESS_1", "WELLNESS_11"]
d["optimism"] = d[optim_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)

# Preferred social integration scale from codebook (SOCINT_MEAN).
# Note: this variable has a known construction issue in the codebook and should be
# interpreted with caution in results text.
d["soc_integration"] = pd.to_numeric(d["SOCINT_MEAN"], errors="coerce")

# Keep a backup integration measure from item-level reconstruction used previously.
soc1 = pd.to_numeric(d["SOC_INTEGRATION_1"], errors="coerce")
soc2 = pd.to_numeric(d["SOC_INTEGRATION_2"], errors="coerce")
soc3 = pd.to_numeric(d["SOC_INTEGRATION_3"], errors="coerce")
d["soc_integration_alt"] = pd.concat([(8 - soc1), soc2, soc3], axis=1).mean(axis=1)

help_cols = ["HELP_SEEKING_1", "HELP_SEEKING_2", "HELP_SEEKING_3"]
d["help_seeking"] = d[help_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)

# Moral injury: item-focused (guilt, shame, betrayal, values violation proxy items)
moral_cols = ["MISSSF_1", "MISSSF_2", "MISSSF_3", "MISSSF_4"]
d["moral_injury"] = d[moral_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
d["moral_injury_total"] = pd.to_numeric(d["MISSSF_TOTAL"], errors="coerce")

# Purpose / meaning
d["purpose_mean"] = d[["PURPOSE_1", "PURPOSE_2", "PURPOSE_3", "PURPOSE_4", "PURPOSE_5"]].apply(
    pd.to_numeric, errors="coerce"
).mean(axis=1)

# Stigma / distancing
d["stigma_mean"] = pd.to_numeric(d["DD_MEAN"], errors="coerce")

# Structural resource indicators (WHOQOL + social/economic)
d["who_safety"] = pd.to_numeric(d["WHO8"], errors="coerce")
d["who_financial_sufficiency"] = pd.to_numeric(d["WHO12"], errors="coerce")
d["who_healthcare_access"] = pd.to_numeric(d["WHO24"], errors="coerce")
d["who_relationship_satisfaction"] = pd.to_numeric(d["WHO20"], errors="coerce")

d["housing"] = pd.to_numeric(d["HOUSING"], errors="coerce")
d["employed"] = pd.to_numeric(d["EMPLOYED"], errors="coerce")
d["occupation"] = pd.to_numeric(d["OCCUPATION"], errors="coerce")

# Confounders
d["substance_risk"] = pd.to_numeric(d["CAGEAID_TOTAL"], errors="coerce")
d["tbi"] = pd.to_numeric(d["TBI"], errors="coerce")

# --- Sub-indices (z-sums) ---
d["risk_zsum"] = zscore(d["combat"]) + zscore(d["stress_count"]) + zscore(d["widowed"])

d["distress_zsum"] = zscore(d["WHO26"]) + zscore(d["sbq_total"])

prot_parts = pd.concat(
    [
        zscore(d["social_support"]),
        zscore(d["optimism"]),
        zscore(d["soc_integration"]),
        zscore(d["help_seeking"]),
    ],
    axis=1,
)
d["protective_zsum"] = prot_parts.sum(axis=1)

d["PGD_vulnerability"] = d["risk_zsum"] + d["distress_zsum"] - d["protective_zsum"]

# Core burden without daily stressors (for moderation model — avoids putting z_stress on both sides)
d["PGD_core"] = (
    zscore(d["combat"])
    + zscore(d["widowed"])
    + zscore(d["WHO26"])
    + zscore(d["sbq_total"])
    - d["protective_zsum"]
)

# Moderators (global z-scores)
d["z_stress"] = zscore(d["stress_count"])
d["support_raw"] = d[["social_support", "optimism", "soc_integration", "help_seeking"]].mean(axis=1)
d["z_support"] = zscore(d["support_raw"])
d["z_friend"] = zscore(pd.to_numeric(d["WHO22"], errors="coerce"))

# --- Regression covariates ---
d["AGE"] = pd.to_numeric(d["AGE"], errors="coerce")
d["BRANCH"] = pd.to_numeric(d["BRANCH"], errors="coerce")
d["DURATION"] = pd.to_numeric(d["DURATION"], errors="coerce")
d["RACE"] = pd.to_numeric(d["RACE"], errors="coerce")
d["INCOME"] = pd.to_numeric(d["INCOME"], errors="coerce")

# --- Structure block (race attenuation + mechanism models) ---
# Exposure: stress_count (STRESS_COUNT), combat (CES_TOTAL / CES)
# Institutional placement: C(BRANCH)
# Life position: AGE, DURATION, INCOME

# Use a common complete-case sample across nested models (same N when comparing).
mod_cols = [
    "PGD_vulnerability",
    "RACE",
    "AGE",
    "BRANCH",
    "DURATION",
    "widowed",
    "stress_count",
    "combat",
    "loneliness",
    "social_support",
    "soc_integration",
    "optimism",
    "moral_injury",
    "purpose_mean",
    "help_seeking",
    "stigma_mean",
    "housing",
    "employed",
    "who_healthcare_access",
    "INCOME",
    "substance_risk",
    "tbi",
]
mod = d.dropna(subset=mod_cols).copy()
print(f"Complete-case analysis rows (hierarchical models): {len(mod):,}")

print(mod[["PGD_vulnerability", "PGD_core", "risk_zsum", "distress_zsum", "protective_zsum"]].describe().T)

Complete-case analysis rows (hierarchical models): 1,469
                    count      mean       std        min       25%       50%  \
PGD_vulnerability  1469.0  0.020310  4.769627  -9.586653 -3.519295 -0.663869   
PGD_core           1469.0  0.016883  4.398174  -8.696491 -3.190380 -0.760378   
risk_zsum          1469.0  0.006246  1.841812  -1.670031 -1.670031 -0.421048   
distress_zsum      1469.0  0.001434  1.785190  -1.909724 -0.895792 -0.626116   
protective_zsum    1469.0 -0.012630  2.931978 -10.620127 -1.806345  0.274277   

                        75%        max  
PGD_vulnerability  2.890413  21.219958  
PGD_core           2.607818  21.485628  
risk_zsum          0.827936   8.971410  
distress_zsum      0.657491   7.269840  
protective_zsum    2.082268   6.006898  


In [27]:
# --- Nested race models (identity → structure) ---
# Model 1: Identity only — raw race/ethnicity differences (reference: RACE code 1)
m1 = smf.ols("PGD_vulnerability ~ C(RACE)", data=mod).fit(cov_type="HC1")
print("Model 1 (identity): PGD_vulnerability ~ C(RACE)")
print(m1.summary())

Model 1 (identity): PGD_vulnerability ~ C(RACE)
                            OLS Regression Results                            
Dep. Variable:      PGD_vulnerability   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                  0.008
Method:                 Least Squares   F-statistic:                     4.031
Date:                Thu, 23 Apr 2026   Prob (F-statistic):            0.00123
Time:                        16:31:37   Log-Likelihood:                -4370.8
No. Observations:                1469   AIC:                             8754.
Df Residuals:                    1463   BIC:                             8785.
Df Model:                           5                                         
Covariance Type:                  HC1                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------


In [28]:
# Model 2: Add structure block — age, branch, tenure, income
# Compare C(RACE) to Model 1: shrinkage / loss of significance suggests structure overlaps with raw gaps.
m2 = smf.ols(
    "PGD_vulnerability ~ C(RACE) + AGE + C(BRANCH) + DURATION + INCOME",
    data=mod,
).fit(cov_type="HC1")
print("Model 2 (identity + structure): PGD_vulnerability ~ C(RACE) + AGE + C(BRANCH) + DURATION + INCOME")
print(m2.summary())


Model 2 (identity + structure): PGD_vulnerability ~ C(RACE) + AGE + C(BRANCH) + DURATION + INCOME
                            OLS Regression Results                            
Dep. Variable:      PGD_vulnerability   R-squared:                       0.068
Model:                            OLS   Adj. R-squared:                  0.059
Method:                 Least Squares   F-statistic:                     7.612
Date:                Thu, 23 Apr 2026   Prob (F-statistic):           1.12e-15
Time:                        16:31:46   Log-Likelihood:                -4326.9
No. Observations:                1469   AIC:                             8684.
Df Residuals:                    1454   BIC:                             8763.
Df Model:                          14                                         
Covariance Type:                  HC1                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------

In [29]:
# Model 3 (mechanisms): what predicts PGD when race is not in the model?
# stress_count = endorsed daily stressor domains; combat = CES_TOTAL
m3_mechanisms = smf.ols(
    "PGD_vulnerability ~ stress_count + combat + social_support + optimism + loneliness + widowed",
    data=mod,
).fit(cov_type="HC1")
print(
    "Model 3 (mechanisms): PGD_vulnerability ~ stress_count + combat + social_support + optimism + loneliness + widowed"
)
print(m3_mechanisms.summary())
print(
    "\nNote: social_support = mean(WHO20, WHO22). High R² can partly reflect overlap with how PGD_vulnerability is built; interpret as associational."
)


Model 3 (mechanisms): PGD_vulnerability ~ stress_count + combat + social_support + optimism + loneliness + widowed
                            OLS Regression Results                            
Dep. Variable:      PGD_vulnerability   R-squared:                       0.832
Model:                            OLS   Adj. R-squared:                  0.831
Method:                 Least Squares   F-statistic:                     1107.
Date:                Thu, 23 Apr 2026   Prob (F-statistic):               0.00
Time:                        16:32:08   Log-Likelihood:                -3070.6
No. Observations:                1469   AIC:                             6155.
Df Residuals:                    1462   BIC:                             6192.
Df Model:                           6                                         
Covariance Type:                  HC1                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
------------

In [31]:
# Exploratory: does the composite protective z-score predict loneliness / optimism?
# z_support = z-score of mean(social_support, optimism, soc_integration, help_seeking)

m_lonely_support = smf.ols("loneliness ~ z_support", data=mod).fit(cov_type="HC1")
print("loneliness ~ z_support")
print(m_lonely_support.summary())

m_optim_support = smf.ols("optimism ~ z_support", data=mod).fit(cov_type="HC1")
print("\noptimism ~ z_support")
print(m_optim_support.summary())

print(
    "\nCaution: optimism is a component of z_support, so optimism ~ z_support is partly tautological; "
    "interpret the slope as alignment within the composite, not an independent predictor test."
)


loneliness ~ z_support
                            OLS Regression Results                            
Dep. Variable:             loneliness   R-squared:                       0.211
Model:                            OLS   Adj. R-squared:                  0.211
Method:                 Least Squares   F-statistic:                     363.8
Date:                Thu, 23 Apr 2026   Prob (F-statistic):           1.26e-72
Time:                        16:45:03   Log-Likelihood:                -1227.2
No. Observations:                1469   AIC:                             2458.
Df Residuals:                    1467   BIC:                             2469.
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.6290      0.

In [30]:
# Model 3: race × stress and race × support (DV excludes stressors so z_stress is not collinear with PGD_core)
m3 = smf.ols(
    "PGD_core ~ C(RACE) * z_stress + C(RACE) * z_support",
    data=mod,
).fit(cov_type="HC1")
print(m3.summary())

                            OLS Regression Results                            
Dep. Variable:               PGD_core   R-squared:                       0.740
Model:                            OLS   Adj. R-squared:                  0.737
Method:                 Least Squares   F-statistic:                     241.6
Date:                Thu, 23 Apr 2026   Prob (F-statistic):               0.00
Time:                        16:32:56   Log-Likelihood:                -3270.0
No. Observations:                1469   AIC:                             6576.
Df Residuals:                    1451   BIC:                             6671.
Df Model:                          17                                         
Covariance Type:                  HC1                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 -0

In [14]:
# Sensitivity: same moderation idea but "support" = friend support (WHO22) only —
# less entangled with the protective z-sum that enters PGD_core.
m3b = smf.ols(
    "PGD_core ~ C(RACE) * z_stress + C(RACE) * z_friend",
    data=mod,
).fit(cov_type="HC1")
print(m3b.summary())

                            OLS Regression Results                            
Dep. Variable:               PGD_core   R-squared:                       0.449
Model:                            OLS   Adj. R-squared:                  0.442
Method:                 Least Squares   F-statistic:                     62.78
Date:                Thu, 23 Apr 2026   Prob (F-statistic):          6.64e-160
Time:                        16:04:57   Log-Likelihood:                -3822.2
No. Observations:                1469   AIC:                             7680.
Df Residuals:                    1451   BIC:                             7776.
Df Model:                          17                                         
Covariance Type:                  HC1                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                -0.00

In [15]:
# --- Hierarchical PGD models (HC1 robust SE) ---
# Model 1: Exposure only (baseline)
m1_exposure = smf.ols(
    "PGD_vulnerability ~ stress_count + combat",
    data=mod,
).fit(cov_type="HC1")

print("Model 1 (Exposure only): PGD_vulnerability ~ stress_count + combat")
print(m1_exposure.summary())

Model 1 (Exposure only): PGD_vulnerability ~ stress_count + combat
                            OLS Regression Results                            
Dep. Variable:      PGD_vulnerability   R-squared:                       0.243
Model:                            OLS   Adj. R-squared:                  0.242
Method:                 Least Squares   F-statistic:                     242.8
Date:                Thu, 23 Apr 2026   Prob (F-statistic):           8.42e-92
Time:                        16:04:57   Log-Likelihood:                -4174.6
No. Observations:                1469   AIC:                             8355.
Df Residuals:                    1466   BIC:                             8371.
Df Model:                           2                                         
Covariance Type:                  HC1                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------

In [16]:
# Model 2: Add social factors
m2_social = smf.ols(
    "PGD_vulnerability ~ stress_count + combat + loneliness + social_support + soc_integration + optimism",
    data=mod,
).fit(cov_type="HC1")

print(
    "\nModel 2 (Social factors): PGD_vulnerability ~ stress_count + combat + loneliness + social_support + soc_integration + optimism"
)
print(m2_social.summary())


Model 2 (Social factors): PGD_vulnerability ~ stress_count + combat + loneliness + social_support + soc_integration + optimism
                            OLS Regression Results                            
Dep. Variable:      PGD_vulnerability   R-squared:                       0.849
Model:                            OLS   Adj. R-squared:                  0.848
Method:                 Least Squares   F-statistic:                     1243.
Date:                Thu, 23 Apr 2026   Prob (F-statistic):               0.00
Time:                        16:04:57   Log-Likelihood:                -2989.8
No. Observations:                1469   AIC:                             5994.
Df Residuals:                    1462   BIC:                             6031.
Df Model:                           6                                         
Covariance Type:                  HC1                                         
                      coef    std err          z      P>|z|      [0.025      0.975

In [17]:
# Model 3: Add psychological processing (moral injury, purpose, help-seeking, stigma)
m3_psych = smf.ols(
    "PGD_vulnerability ~ stress_count + combat + loneliness + social_support + soc_integration + optimism + moral_injury + purpose_mean + help_seeking + stigma_mean",
    data=mod,
).fit(cov_type="HC1")

print(
    "\nModel 3 (Psych processing): PGD_vulnerability ~ stress_count + combat + loneliness + social_support + soc_integration + optimism + moral_injury + purpose_mean + help_seeking + stigma_mean"
)
print(m3_psych.summary())


Model 3 (Psych processing): PGD_vulnerability ~ stress_count + combat + loneliness + social_support + soc_integration + optimism + moral_injury + purpose_mean + help_seeking + stigma_mean
                            OLS Regression Results                            
Dep. Variable:      PGD_vulnerability   R-squared:                       0.881
Model:                            OLS   Adj. R-squared:                  0.880
Method:                 Least Squares   F-statistic:                     972.9
Date:                Thu, 23 Apr 2026   Prob (F-statistic):               0.00
Time:                        16:04:57   Log-Likelihood:                -2814.1
No. Observations:                1469   AIC:                             5650.
Df Residuals:                    1458   BIC:                             5708.
Df Model:                          10                                         
Covariance Type:                  HC1                                         
                     

In [18]:
# Model 4: Add structural factors and key confounders
m4_structural = smf.ols(
    "PGD_vulnerability ~ stress_count + combat + loneliness + social_support + soc_integration + optimism + moral_injury + purpose_mean + help_seeking + stigma_mean + INCOME + housing + who_healthcare_access + employed + substance_risk + tbi",
    data=mod,
).fit(cov_type="HC1")

print(
    "\nModel 4 (Structural): PGD_vulnerability ~ stress_count + combat + loneliness + social_support + soc_integration + optimism + moral_injury + purpose_mean + help_seeking + stigma_mean + INCOME + housing + who_healthcare_access + employed + substance_risk + tbi"
)
print(m4_structural.summary())

print(
    "\nNote: soc_integration uses SOCINT_MEAN. The codebook flags a known construction issue for this variable, so interpret that coefficient cautiously and report this limitation explicitly."
)


Model 4 (Structural): PGD_vulnerability ~ stress_count + combat + loneliness + social_support + soc_integration + optimism + moral_injury + purpose_mean + help_seeking + stigma_mean + INCOME + housing + who_healthcare_access + employed + substance_risk + tbi
                            OLS Regression Results                            
Dep. Variable:      PGD_vulnerability   R-squared:                       0.888
Model:                            OLS   Adj. R-squared:                  0.887
Method:                 Least Squares   F-statistic:                     640.9
Date:                Thu, 23 Apr 2026   Prob (F-statistic):               0.00
Time:                        16:04:58   Log-Likelihood:                -2771.8
No. Observations:                1469   AIC:                             5578.
Df Residuals:                    1452   BIC:                             5668.
Df Model:                          16                                         
Covariance Type:             

In [19]:
# PGD_core: stress × loneliness (formula has no loneliness main effect — only z_stress:loneliness)
m_core_stress_lonely = smf.ols(
    "PGD_core ~ z_stress + social_support + optimism + z_stress:loneliness",
    data=mod,
).fit(cov_type="HC1")
print(
    "PGD_core ~ z_stress + social_support + optimism + z_stress:loneliness (HC1)"
)
print(m_core_stress_lonely.summary())

PGD_core ~ z_stress + social_support + optimism + z_stress:loneliness (HC1)
                            OLS Regression Results                            
Dep. Variable:               PGD_core   R-squared:                       0.693
Model:                            OLS   Adj. R-squared:                  0.693
Method:                 Least Squares   F-statistic:                     742.2
Date:                Thu, 23 Apr 2026   Prob (F-statistic):               0.00
Time:                        16:04:58   Log-Likelihood:                -3391.6
No. Observations:                1469   AIC:                             6793.
Df Residuals:                    1464   BIC:                             6820.
Df Model:                           4                                         
Covariance Type:                  HC1                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------

In [20]:
# Support / optimism by race + income + branch + age
mod_so = mod.copy()
mod_so["INCOME"] = pd.to_numeric(mod_so["INCOME"], errors="coerce")

m_support = smf.ols(
    "social_support ~ C(RACE) + INCOME + C(BRANCH) + AGE",
    data=mod_so,
).fit(cov_type="HC1")

m_optimism = smf.ols(
    "optimism ~ C(RACE) + INCOME + C(BRANCH) + AGE",
    data=mod_so,
).fit(cov_type="HC1")

print("social_support ~ C(RACE) + INCOME + C(BRANCH) + AGE")
print(m_support.summary())

social_support ~ C(RACE) + INCOME + C(BRANCH) + AGE
                            OLS Regression Results                            
Dep. Variable:         social_support   R-squared:                       0.016
Model:                            OLS   Adj. R-squared:                  0.007
Method:                 Least Squares   F-statistic:                     1.713
Date:                Thu, 23 Apr 2026   Prob (F-statistic):             0.0525
Time:                        16:04:58   Log-Likelihood:                -1964.0
No. Observations:                1469   AIC:                             3956.
Df Residuals:                    1455   BIC:                             4030.
Df Model:                          13                                         
Covariance Type:                  HC1                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------

In [21]:
print(m_optimism.summary())

                            OLS Regression Results                            
Dep. Variable:               optimism   R-squared:                       0.027
Model:                            OLS   Adj. R-squared:                  0.018
Method:                 Least Squares   F-statistic:                     3.676
Date:                Thu, 23 Apr 2026   Prob (F-statistic):           8.97e-06
Time:                        16:04:58   Log-Likelihood:                -1942.5
No. Observations:                1469   AIC:                             3913.
Df Residuals:                    1455   BIC:                             3987.
Df Model:                          13                                         
Covariance Type:                  HC1                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          3.3336      0.110     30.

In [22]:
# Vulnerability by support/optimism + race
m_vuln_support_optimism_race = smf.ols(
    "PGD_vulnerability ~ social_support + optimism + C(RACE)",
    data=mod_so,
).fit(cov_type="HC1")

print("\nPGD_vulnerability ~ social_support + optimism + C(RACE)")
print(m_vuln_support_optimism_race.summary())


PGD_vulnerability ~ social_support + optimism + C(RACE)
                            OLS Regression Results                            
Dep. Variable:      PGD_vulnerability   R-squared:                       0.650
Model:                            OLS   Adj. R-squared:                  0.649
Method:                 Least Squares   F-statistic:                     348.1
Date:                Thu, 23 Apr 2026   Prob (F-statistic):          6.59e-306
Time:                        16:04:58   Log-Likelihood:                -3607.0
No. Observations:                1469   AIC:                             7230.
Df Residuals:                    1461   BIC:                             7272.
Df Model:                           7                                         
Covariance Type:                  HC1                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------

In [25]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf



# make a working copy
d = df.copy()

# ---------- core variables ----------
# adjust names here if your dataframe uses slightly different column names
d["stress_count"] = pd.to_numeric(d["STRESS_COUNT"], errors="coerce")
d["combat"] = pd.to_numeric(d["CES_TOTAL"], errors="coerce")
d["loneliness"] = pd.to_numeric(d["LON_MEAN"], errors="coerce")

# distress
d["sbq_total"] = pd.to_numeric(d["SBQ_TOTAL"], errors="coerce")
d["who26"] = pd.to_numeric(d["WHO26"], errors="coerce")

# if you already made distress_zsum earlier, you can keep that instead
d["distress_zsum"] = zscore(d["sbq_total"]) + zscore(d["who26"])

# protective pieces
# if you already created these columns earlier, this will just reuse them
if "social_support" not in d.columns:
    d["WHO20"] = pd.to_numeric(d["WHO20"], errors="coerce")
    d["WHO22"] = pd.to_numeric(d["WHO22"], errors="coerce")
    d["social_support"] = d[["WHO20", "WHO22"]].mean(axis=1)

if "optimism" not in d.columns:
    d["WELLNESS_1"] = pd.to_numeric(d["WELLNESS_1"], errors="coerce")
    d["WELLNESS_11"] = pd.to_numeric(d["WELLNESS_11"], errors="coerce")
    d["optimism"] = d[["WELLNESS_1", "WELLNESS_11"]].mean(axis=1)

if "soc_integration" not in d.columns:
    d["SOC_INTEGRATION_1"] = pd.to_numeric(d["SOC_INTEGRATION_1"], errors="coerce")
    d["SOC_INTEGRATION_2"] = pd.to_numeric(d["SOC_INTEGRATION_2"], errors="coerce")
    d["SOC_INTEGRATION_3"] = pd.to_numeric(d["SOC_INTEGRATION_3"], errors="coerce")
    # reverse item 1 so higher = more integration
    d["soc1_rev"] = 8 - d["SOC_INTEGRATION_1"]
    d["soc_integration"] = d[["soc1_rev", "SOC_INTEGRATION_2", "SOC_INTEGRATION_3"]].mean(axis=1)

# ---------- build separate indices ----------
# risk index: stress + combat + distress + loneliness
d["risk_index"] = (
    zscore(d["stress_count"])
    + zscore(d["combat"])
    + zscore(d["distress_zsum"])
    + zscore(d["loneliness"])
)

# protective index: support + optimism + integration
d["protective_index"] = (
    zscore(d["social_support"])
    + zscore(d["optimism"])
    + zscore(d["soc_integration"])
)

# optional: overall net vulnerability = risk - protection
d["pgd_balance_index"] = d["risk_index"] - d["protective_index"]

# ---------- inspect ----------
print(d[["risk_index", "protective_index", "pgd_balance_index"]].describe())

# ---------- test each separately ----------
# example covariates; remove/add as needed
cols_needed = [
    "risk_index", "protective_index", "stress_count", "combat", "loneliness",
    "social_support", "optimism", "soc_integration",
    "RACE", "AGE", "BRANCH", "DURATION", "INCOME"
]

mod = d.copy()

# risk as outcome
m_risk = smf.ols(
    "risk_index ~ C(RACE) + AGE + C(BRANCH) + DURATION + INCOME",
    data=mod
).fit(cov_type="HC1")

print("\nRisk index model")
print(m_risk.summary())

# protection as outcome
m_protect = smf.ols(
    "protective_index ~ C(RACE) + AGE + C(BRANCH) + DURATION + INCOME",
    data=mod
).fit(cov_type="HC1")

print("\nProtective index model")
print(m_protect.summary())

# ---------- include both in one regression ----------
# If you want to predict your existing PGD_vulnerability variable:
m_both = smf.ols(
    "pgd_balance_index ~ risk_index + protective_index + C(RACE) + AGE + C(BRANCH) + DURATION + INCOME",
    data=mod
).fit(cov_type="HC1")

print("\nPGD_vulnerability ~ risk_index + protective_index")
print(m_both.summary())

# ---------- optional simpler model ----------
m_both_simple = smf.ols(
    "pgd_balance_index ~ risk_index + protective_index",
    data=mod
).fit(cov_type="HC1")

print("\nPGD_vulnerability ~ risk_index + protective_index (simple)")
print(m_both_simple.summary())

        risk_index  protective_index  pgd_balance_index
count  1469.000000      1.495000e+03        1469.000000
mean      0.005737     -1.045615e-16           0.018989
std       2.777652      2.498705e+00           4.540946
min      -3.569258     -8.140689e+00          -8.071847
25%      -2.357073     -1.597978e+00          -3.472593
50%      -0.592527      2.504253e-01          -0.673292
75%       1.845705      1.769898e+00           2.967217
max      10.508630      4.502589e+00          16.164386

Risk index model
                            OLS Regression Results                            
Dep. Variable:             risk_index   R-squared:                       0.246
Model:                            OLS   Adj. R-squared:                  0.239
Method:                 Least Squares   F-statistic:                     33.50
Date:                Thu, 23 Apr 2026   Prob (F-statistic):           2.47e-78
Time:                        16:06:24   Log-Likelihood:                -3377.4
No. 